# Mystic Camp — Historical Stage & Peak Review (2019–2026)
**Station:** USGS 08165500 · **Units:** feet · **Times:** UTC

This notebook **only reads** the Parquet outputs from `examples/analyze_mystic_camp_history.py`. It does not change peak detection, threshold calculations, or the original records.

**Navigation:** Run cells in order. Plotly charts support zooming, panning, hovering, and legend toggling. The full historical plot is downsampled **for display only**; all peak markers and all observations in the detailed plots are retained.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

ROOT = Path.home() / "Documents" / "usgs_mrms_research"
DATA_DIR = ROOT / "data/stage_peaks_historical_2019_2026/08165500"
FILES = {
    "stage": "filtered_stage.parquet",
    "top200": "top_200_peaks.parquet",
    "retained": "filtered_peaks.parquet",
    "events": "detected_events.parquet",
}
for name, filename in FILES.items():
    path = DATA_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Missing {name}: {path}\nRun examples/analyze_mystic_camp_history.py first.")

stage = pd.read_parquet(DATA_DIR / FILES["stage"])
top200 = pd.read_parquet(DATA_DIR / FILES["top200"])
retained = pd.read_parquet(DATA_DIR / FILES["retained"])
events = pd.read_parquet(DATA_DIR / FILES["events"])

print("Data directory:", DATA_DIR)
for name, df in [("Stage observations", stage), ("Top 200", top200), ("Retained peaks", retained), ("Detected events", events)]:
    print(f"{name}: {len(df):,} rows")
    print("  columns:", ", ".join(df.columns))


Data directory: /Users/gonzalo/Documents/usgs_mrms_research/data/stage_peaks_historical_2019_2026/08165500
Stage observations: 784,452 rows
  columns: datetime, Stage_ft, time_difference, new_segment, segment_id, baseline_ft, quick_stage_ft, filter_applied
Top 200: 200 rows
  columns: selected_rank, segment_id, i_start, i_peak, i_end, date_start, flow_start, baseflow_start, date_peak, flow_peak, baseflow_peak, date_end, flow_end, baseflow_end, month_num, month_name, event_volume, baseflow_volume, candidate_type, baseline_available
Retained peaks: 127 rows
  columns: selected_rank, segment_id, i_start, i_peak, i_end, date_start, flow_start, baseflow_start, date_peak, flow_peak, baseflow_peak, date_end, flow_end, baseflow_end, month_num, month_name, event_volume, baseflow_volume, candidate_type, baseline_available
Detected events: 58,847 rows
  columns: segment_id, i_start, i_peak, i_end, date_start, flow_start, baseflow_start, date_peak, flow_peak, baseflow_peak, date_end, flow_end, bas

## 1. Identify columns and prepare timestamps
The notebook checks the actual columns rather than assuming a particular timestamp or baseline name.

In [3]:
def choose_column(df, choices, label, required=True):
    match = next((c for c in choices if c in df.columns), None)
    if match is None and required:
        raise KeyError(f"Cannot identify {label}. Available: {list(df.columns)}")
    return match

time_col = choose_column(stage, ["datetime", "date", "timestamp"], "observation time")
stage_col = choose_column(stage, ["Stage_ft", "stage_ft", "flow", "streamflow"], "observed stage")
baseline_col = choose_column(stage, ["baseflow", "baseflow_ft", "Baseflow_ft", "baseflow_stage_ft"], "baseline", required=False)
for df, col in [(stage, time_col), (top200, "date_peak"), (retained, "date_peak")]:
    df[col] = pd.to_datetime(df[col], utc=True)
stage = stage.sort_values(time_col).reset_index(drop=True)

print("Observation time:", time_col)
print("Observed stage:", stage_col)
print("Baseline:", baseline_col or "not found; baseline trace will be omitted")
print("Period:", stage[time_col].min(), "to", stage[time_col].max())
print("Maximum observed stage:", stage[stage_col].max(), "ft")


Observation time: datetime
Observed stage: Stage_ft
Baseline: not found; baseline trace will be omitted
Period: 2019-01-01 00:00:00+00:00 to 2026-10-16 23:55:00+00:00
Maximum observed stage: 37.52 ft


## 2. Entire history — observed stage and selected peaks
The observed series is sampled to approximately 20,000 points **only in this overview** to keep the chart responsive. The original 200 and retained peaks are all displayed, including the 37.52 ft record.

In [4]:
def marker_trace(df, label, color, symbol, size=8):
    return go.Scattergl(
        x=df["date_peak"], y=df["flow_peak"], mode="markers", name=label,
        marker=dict(color=color, size=size, symbol=symbol),
        customdata=df["candidate_type"].astype(str) if "candidate_type" in df else None,
        hovertemplate="%{x}<br>Stage: %{y:.2f} ft<extra>" + label + "</extra>",
    )

step = max(1, int(np.ceil(len(stage) / 20000)))
overview = stage.iloc[::step]
fig = go.Figure()
fig.add_trace(go.Scattergl(x=overview[time_col], y=overview[stage_col], mode="lines", name="Observed stage (display sample)", line=dict(color="#64748b", width=1)))
fig.add_trace(marker_trace(top200, "Top 200 candidates", "#f59e0b", "circle", 7))
fig.add_trace(marker_trace(retained, "Retained after threshold", "#dc2626", "circle", 9))
fig.update_layout(title="Mystic Camp | Historical stage and peak candidates", xaxis_title="Date (UTC)", yaxis_title="Stage (ft)", height=650, hovermode="closest", template="plotly_white", legend=dict(orientation="h", y=1.08))
fig.show()


## 3. July 4, 2025 — full-resolution flood inspection
Change the start and end dates below to explore other periods. **No downsampling** is applied to this detailed view. Missing observations are not interpolated; lines are broken when the time gap exceeds 20 minutes.

In [5]:
START = "2025-07-03 00:00:00+00:00"
END   = "2025-07-06 00:00:00+00:00"

def plot_window(start, end, show_baseline=True):
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    if start.tzinfo is None: start = start.tz_localize("UTC")
    if end.tzinfo is None: end = end.tz_localize("UTC")
    if end <= start: raise ValueError("END must be after START")
    subset = stage.loc[stage[time_col].between(start, end)].copy()
    if subset.empty:
        print("No observations in the selected window")
        return
    # Split lines at discontinuities, while keeping every original observation.
    gap = subset[time_col].diff().gt(pd.Timedelta(minutes=20))
    observed = subset[stage_col].astype(float).copy()
    observed.loc[gap] = np.nan
    fig = go.Figure()
    fig.add_trace(go.Scattergl(x=subset[time_col], y=observed, mode="lines", name="Observed stage (continuous portions)", line=dict(color="#2563eb", width=2)))
    fig.add_trace(go.Scattergl(x=subset[time_col], y=subset[stage_col], mode="markers", name="Original observations", marker=dict(color="#2563eb", size=4, opacity=0.55)))
    if show_baseline and baseline_col:
        baseline = pd.to_numeric(subset[baseline_col], errors="coerce").copy()
        baseline.loc[gap] = np.nan
        fig.add_trace(go.Scattergl(x=subset[time_col], y=baseline, mode="lines", name="Lyne–Hollick baseline", line=dict(color="#16a34a", width=2)))
    for peaks, label, color in [(top200, "Top 200 candidates", "#f59e0b"), (retained, "Retained peaks", "#dc2626")]:
        p = peaks.loc[peaks["date_peak"].between(start, end)]
        fig.add_trace(marker_trace(p, label, color, "circle", 11))
    fig.update_layout(title=f"Mystic Camp | {start:%Y-%m-%d} to {end:%Y-%m-%d} (UTC)", xaxis_title="Date (UTC)", yaxis_title="Stage (ft)", template="plotly_white", height=680, hovermode="closest", legend=dict(orientation="h", y=1.1))
    fig.show()
    print(f"Original observations: {len(subset):,} | Max observed: {subset[stage_col].max():.2f} ft")
    display(top200.loc[top200["date_peak"].between(start, end), [c for c in ["selected_rank", "date_peak", "flow_peak", "baseflow_peak", "candidate_type"] if c in top200.columns]].sort_values("flow_peak", ascending=False))

plot_window(START, END)


Original observations: 446 | Max observed: 37.52 ft


,selected_rank,date_peak,flow_peak,baseflow_peak,candidate_type
0,1,2025-07-04 10:10:00+00:00,37.52,NaN,unfiltered_observation
1,2,2025-07-04 09:35:00+00:00,29.45,19.916830,segment_boundary_or_maximum
16,17,2025-07-05 20:35:00+00:00,8.89,8.836167,segment_boundary_or_maximum


## 4. Compare the top 200 with retained peaks
This plot shows the **candidate threshold** as reported by the previous run (8.2567 ft). It is a reference annotation, not a recalculation. Check `candidate_threshold.json` for the exact value and metadata.

In [9]:

# ============================================================
# CELL 10 — TOP 200 VS RETAINED PEAKS
# Statistical candidate threshold visualization
# ============================================================

import json
import plotly.graph_objects as go
from IPython.display import display

# ------------------------------------------------------------
# 1. Load the existing threshold metadata
# ------------------------------------------------------------

threshold_file = DATA_DIR / "candidate_threshold.json"

if not threshold_file.is_file():
    raise FileNotFoundError(threshold_file)

with threshold_file.open("r", encoding="utf-8") as file:
    threshold_data = json.load(file)

print("=" * 70)
print("STATISTICAL CANDIDATE THRESHOLD")
print("=" * 70)

print(json.dumps(threshold_data, indent=2, default=str))

# ------------------------------------------------------------
# 2. Extract the correct threshold
# ------------------------------------------------------------

threshold = threshold_data.get(
    "significant_stage_threshold_candidate_ft"
)

if threshold is None:
    raise ValueError(
        "The statistical candidate threshold is missing "
        "from candidate_threshold.json."
    )

threshold = float(threshold)

print(f"\nCandidate threshold: {threshold:.4f} ft")
print(f"Original peak candidates: {len(top200)}")
print(f"Retained peak candidates: {len(retained)}")
print(f"Excluded candidates: {len(top200) - len(retained)}")

# ------------------------------------------------------------
# 3. Plot top 200 and retained candidates
# ------------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    marker_trace(
        top200,
        "Top 200 candidates",
        "#f59e0b",
        "circle",
        8,
    )
)

fig.add_trace(
    marker_trace(
        retained,
        "Retained after threshold",
        "#dc2626",
        "circle",
        9,
    )
)

# ------------------------------------------------------------
# 4. Statistical candidate threshold reference
# ------------------------------------------------------------

fig.add_hline(
    y=threshold,
    line_dash="dash",
    line_color="#16a34a",
    line_width=2,
    annotation_text=f"Candidate threshold: {threshold:.4f} ft",
    annotation_position="top right",
)

# ------------------------------------------------------------
# 5. Figure layout
# ------------------------------------------------------------

fig.update_layout(
    title=(
        "USGS 08165500 — Mystic Camp"
        "<br>Top 200 vs Retained Peak Candidates"
    ),
    xaxis_title="Peak date (UTC)",
    yaxis_title="Observed water stage (ft)",
    template="plotly_white",
    height=650,
    hovermode="closest",
    legend=dict(
        orientation="h",
        y=1.12,
        x=0,
    ),
    xaxis=dict(
        rangeslider=dict(visible=True)
    ),
)

fig.show()

# ------------------------------------------------------------
# 6. Candidate type summary
# ------------------------------------------------------------

print("\nCANDIDATE TYPES — TOP 200")

display(
    top200["candidate_type"]
    .value_counts()
    .rename_axis("candidate_type")
    .to_frame("count")
)

print("\nVisualization completed.")



STATISTICAL CANDIDATE THRESHOLD
{
  "status": "ok",
  "number_of_events": 200,
  "number_of_valid_baseflows": 199,
  "window_width_ft": 1.0,
  "window_min_ft": 7.8197528245676216,
  "window_max_ft": 8.797635825678519,
  "events_in_window": 189,
  "fraction_in_window": 0.949748743718593,
  "median_all_baseflow_ft": 8.263059171064555,
  "median_window_baseflow_ft": 8.25671154232561,
  "tie_breaking": "first densest interval in ascending baseflow order",
  "threshold_method": "median_baseflow_densest_1ft_window_top_200_peaks",
  "significant_stage_threshold_candidate_ft": 8.25671154232561,
  "threshold_for_peak_filter_ft": 8.25671154232561,
  "threshold_peak_filter_rule": "observed_peak_stage_ft >= threshold_for_peak_filter_ft",
  "threshold_mrms_validated": false,
  "validation_status": "candidate_for_testing_only",
  "filtered_peak_count": 127
}

Candidate threshold: 8.2567 ft
Original peak candidates: 200
Retained peak candidates: 127
Excluded candidates: 73



CANDIDATE TYPES — TOP 200


,count
candidate_type,
detected_event,195
segment_boundary_or_maximum,4
unfiltered_observation,1



Visualization completed.


## 5. Peak height and baseline distributions
Histograms are based on the saved peak tables. Missing baselines remain missing; no values are imputed.

In [10]:
from plotly.subplots import make_subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=("Observed peak stage (ft)", "Baseline at peak (ft)"))
fig.add_trace(go.Histogram(x=top200["flow_peak"], name="Top 200", marker_color="#f59e0b", opacity=0.65), row=1, col=1)
fig.add_trace(go.Histogram(x=retained["flow_peak"], name="Retained", marker_color="#dc2626", opacity=0.65), row=1, col=1)
if "baseflow_peak" in top200.columns:
    fig.add_trace(go.Histogram(x=top200["baseflow_peak"].dropna(), name="Baseline — top 200", marker_color="#16a34a"), row=1, col=2)
fig.update_layout(barmode="overlay", title="Peak and baseline distributions", template="plotly_white", height=480)
fig.show()


## 6. Confirm the original maximum observation is preserved
This checks that the observed maximum appears among the selected peaks, without changing the algorithm.

In [8]:
maximum = float(stage[stage_col].max())
max_rows = stage.loc[np.isclose(pd.to_numeric(stage[stage_col]), maximum), [time_col, stage_col]]
matched = top200.loc[np.isclose(pd.to_numeric(top200["flow_peak"]), maximum)]
print(f"Maximum original observation: {maximum:.2f} ft")
print(f"Matching rows in top 200: {len(matched)}")
display(max_rows)
display(matched)
assert len(matched) >= 1, "The original maximum was not found in top 200"


Maximum original observation: 37.52 ft
Matching rows in top 200: 1


,datetime,Stage_ft
649949,2025-07-04 10:10:00+00:00,37.52


,selected_rank,segment_id,i_start,i_peak,i_end,date_start,flow_start,baseflow_start,date_peak,flow_peak,baseflow_peak,date_end,flow_end,baseflow_end,month_num,month_name,event_volume,baseflow_volume,candidate_type,baseline_available
0,1,39,NaN,NaN,NaN,NaT,NaN,NaN,2025-07-04 10:10:00+00:00,37.52,NaN,NaT,NaN,NaN,NaN,None,NaN,NaN,unfiltered_observation,False


## 7. Review another date range
Edit the two timestamps and rerun the cell. You can inspect any event at full resolution without regenerating HTML.

In [ ]:
# Example: 2026 July flood
CUSTOM_START = "2026-07-15 00:00:00+00:00"
CUSTOM_END   = "2026-07-18 23:59:59+00:00"
plot_window(CUSTOM_START, CUSTOM_END)


This cell will analyze the number of retained peaks per year, their observed stage, and their difference from the Lyne–Hollick baseline. It will also identify peaks without baseline estimates and show an interactive Plotly visualization.

In [11]:

# ============================================================
# CELL 11 — SCIENTIFIC VALIDATION OF RETAINED PEAKS
# USGS 08165500 — Mystic Camp
# ============================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

# ------------------------------------------------------------
# 1. Prepare retained candidates
# ------------------------------------------------------------

validation = retained.copy()

validation["date_peak"] = pd.to_datetime(
    validation["date_peak"], utc=True
)

validation["flow_peak"] = pd.to_numeric(
    validation["flow_peak"], errors="coerce"
)

validation["baseflow_peak"] = pd.to_numeric(
    validation["baseflow_peak"], errors="coerce"
)

validation["year"] = validation["date_peak"].dt.year

# Difference between observed stage and filtered baseline.
# Missing baselines remain missing; no interpolation.

validation["stage_above_baseline_ft"] = (
    validation["flow_peak"] - validation["baseflow_peak"]
)

validation["baseline_available"] = np.isfinite(
    validation["baseflow_peak"]
)

validation = validation.sort_values(
    "date_peak"
).reset_index(drop=True)

print("=" * 70)
print("MYSTIC CAMP — RETAINED PEAK VALIDATION")
print("=" * 70)

print("Total retained candidates:", len(validation))
print(
    "Candidates with valid baseline:",
    int(validation["baseline_available"].sum())
)
print(
    "Candidates without baseline:",
    int((~validation["baseline_available"]).sum())
)

# ------------------------------------------------------------
# 2. Distribution by year
# ------------------------------------------------------------

annual = (
    validation.groupby("year")
    .agg(
        retained_peaks=("flow_peak", "size"),
        maximum_stage_ft=("flow_peak", "max"),
        median_stage_ft=("flow_peak", "median"),
        median_stage_above_baseline_ft=(
            "stage_above_baseline_ft", "median"
        ),
    )
    .reset_index()
)

# Include years with no retained candidates.
annual = (
    pd.DataFrame({"year": range(2019, 2027)})
    .merge(annual, on="year", how="left")
)

annual["retained_peaks"] = (
    annual["retained_peaks"].fillna(0).astype(int)
)

print("\nRETAINED PEAKS BY YEAR")
display(annual)

# ------------------------------------------------------------
# 3. Plot annual candidate counts
# ------------------------------------------------------------

fig_year = go.Figure()

fig_year.add_trace(
    go.Bar(
        x=annual["year"].astype(str),
        y=annual["retained_peaks"],
        text=annual["retained_peaks"],
        textposition="outside",
        marker_color="#2563eb",
        name="Retained peaks",
    )
)

fig_year.update_layout(
    title="Mystic Camp — Retained Peak Candidates by Year",
    xaxis_title="Year",
    yaxis_title="Number of retained candidates",
    template="plotly_white",
    height=450,
)

fig_year.show()

# ------------------------------------------------------------
# 4. Observed stage vs filtered baseline
# ------------------------------------------------------------

with_baseline = validation.loc[
    validation["baseline_available"]
].copy()

fig_baseline = go.Figure()

fig_baseline.add_trace(
    go.Scatter(
        x=with_baseline["baseflow_peak"],
        y=with_baseline["flow_peak"],
        mode="markers",
        name="Retained peak candidates",
        marker=dict(
            color=with_baseline["stage_above_baseline_ft"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(
                title="Stage rise (ft)"
            ),
            size=9,
        ),
        customdata=with_baseline[
            ["date_peak"]
        ].astype(str).to_numpy(),
        hovertemplate=(
            "Date: %{customdata[0]}<br>"
            "Baseline: %{x:.3f} ft<br>"
            "Observed peak: %{y:.3f} ft<br>"
            "<extra></extra>"
        ),
    )
)

minimum = min(
    with_baseline["baseflow_peak"].min(),
    with_baseline["flow_peak"].min()
)

maximum = max(
    with_baseline["baseflow_peak"].max(),
    with_baseline["flow_peak"].max()
)

fig_baseline.add_trace(
    go.Scatter(
        x=[minimum, maximum],
        y=[minimum, maximum],
        mode="lines",
        line=dict(
            color="gray",
            dash="dash"
        ),
        name="Observed stage = baseline",
    )
)

fig_baseline.update_layout(
    title="Observed Peak Stage vs Lyne–Hollick Baseline",
    xaxis_title="Filtered baseline at peak (ft)",
    yaxis_title="Observed peak stage (ft)",
    template="plotly_white",
    height=600,
)

fig_baseline.show()

# ------------------------------------------------------------
# 5. Stage increase above baseline
# ------------------------------------------------------------

print("\nSTAGE ABOVE BASELINE STATISTICS")

if not with_baseline.empty:
    print(
        with_baseline["stage_above_baseline_ft"]
        .describe()
        .to_string()
    )
else:
    print("No valid baseline estimates.")

# ------------------------------------------------------------
# 6. Highest retained peaks
# ------------------------------------------------------------

print("\nTOP 20 RETAINED PEAKS")

top_20 = validation.sort_values(
    "flow_peak", ascending=False
).head(20)

display(
    top_20[
        [
            "date_peak",
            "flow_peak",
            "baseflow_peak",
            "stage_above_baseline_ft",
            "candidate_type",
        ]
    ]
)

# ------------------------------------------------------------
# 7. Candidates without baseline
# ------------------------------------------------------------

print("\nRETAINED CANDIDATES WITHOUT BASELINE")

missing_baseline = validation.loc[
    ~validation["baseline_available"]
]

if missing_baseline.empty:
    print("All retained candidates have baseline estimates.")
else:
    display(
        missing_baseline[
            [
                "date_peak",
                "flow_peak",
                "candidate_type",
            ]
        ]
    )

print("\nScientific validation diagnostic completed.")


MYSTIC CAMP — RETAINED PEAK VALIDATION
Total retained candidates: 127
Candidates with valid baseline: 126
Candidates without baseline: 1

RETAINED PEAKS BY YEAR


,year,retained_peaks,maximum_stage_ft,median_stage_ft,median_stage_above_baseline_ft
0,2019,6,9.20,8.405,0.343720
1,2020,0,NaN,NaN,NaN
2,2021,8,9.85,8.415,0.021419
3,2022,0,NaN,NaN,NaN
4,2023,0,NaN,NaN,NaN
5,2024,5,9.75,8.640,0.249242
6,2025,63,37.52,8.330,0.006964
7,2026,45,19.30,8.320,0.004840



STAGE ABOVE BASELINE STATISTICS
count    126.000000
mean       0.299003
std        1.153045
min        0.000000
25%        0.003448
50%        0.006457
75%        0.021695
max        9.533170

TOP 20 RETAINED PEAKS


,date_peak,flow_peak,baseflow_peak,stage_above_baseline_ft,candidate_type
20,2025-07-04 10:10:00+00:00,37.52,NaN,NaN,unfiltered_observation
19,2025-07-04 09:35:00+00:00,29.45,19.916830,9.533170,segment_boundary_or_maximum
85,2026-07-16 08:30:00+00:00,19.30,12.244699,7.055301,detected_event
86,2026-07-16 14:20:00+00:00,18.15,13.619291,4.530709,detected_event
87,2026-07-17 00:00:00+00:00,14.22,12.523205,1.696795,detected_event
33,2025-07-14 23:05:00+00:00,12.97,10.731671,2.238329,detected_event
88,2026-07-17 14:25:00+00:00,10.99,10.343450,0.646550,detected_event
34,2025-07-15 17:00:00+00:00,10.31,9.759747,0.550253,detected_event
22,2025-07-07 00:35:00+00:00,10.13,9.113218,1.016782,detected_event
6,2021-05-01 20:45:00+00:00,9.85,8.780956,1.069044,detected_event



RETAINED CANDIDATES WITHOUT BASELINE


,date_peak,flow_peak,candidate_type
20,2025-07-04 10:10:00+00:00,37.52,unfiltered_observation



Scientific validation diagnostic completed.
